# Session 01 · Exercise 2 — Model response via API (Anthropic)

Model: `claude-haiku-4-5-20251001` · API: Messages API (`client.messages.create`).
Key: Colab Secret `Anthropic-API`. Same question and role as `ex02_openai.ipynb`.

In [ ]:
# Instala el SDK de Anthropic
!pip install anthropic

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("Anthropic-API")

from anthropic import (
    Anthropic,
    AuthenticationError,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    InternalServerError,
)
anthropic_client = Anthropic()  # Lee la key del entorno automáticamente

In [ ]:
# claude-haiku-4-5 pricing (check periodically)
PRICING = {
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
}

def query_claude(message, system_prompt=None, model="claude-haiku-4-5-20251001"):
    """
    Make a call to the Anthropic Messages API and return
    the response along with relevant metadata.
    """
    try:
        kwargs = {
            "model": model,
            "messages": [{"role": "user", "content": message}],
            "max_tokens": 1000,
            # No temperature: anthropic SDK 1.x+ (Colab has 2.8.0) rejects it.
        }
        if system_prompt:
            kwargs["system"] = system_prompt

        response = anthropic_client.messages.create(**kwargs)

        # Check if the response was truncated
        if response.stop_reason == "max_tokens":
            print("⚠️  Response truncated")

        # Calculate cost
        prices = PRICING.get(model, {"input": 0, "output": 0})
        cost = (
            (response.usage.input_tokens / 1_000_000) * prices["input"] +
            (response.usage.output_tokens / 1_000_000) * prices["output"]
        )

        return {
            "content": response.content[0].text,
            "model": response.model,
            "id": response.id,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "stop_reason": response.stop_reason,
            "cost_usd": cost,
        }

    except AuthenticationError:
        return {"error": "Invalid or missing API key"}
    except RateLimitError:
        return {"error": "Rate limit reached or insufficient credit"}
    except BadRequestError as e:
        return {"error": f"Invalid request: {e.message}"}
    except (APIConnectionError, InternalServerError):
        return {"error": "Connection or server error"}


def print_result(title, r):
    """Print the answer and its metadata (Level 3) for one call."""
    print("=" * 60)
    print(title)
    print("=" * 60)
    if "error" in r:
        print(f"Error: {r['error']}")
        return
    print(r["content"])
    print("\n--- Metadata ---")
    print(f"Model: {r['model']}")
    print(f"ID: {r['id']}")
    print(f"Tokens: {r['input_tokens']} input + {r['output_tokens']} output")
    print(f"Stop reason: {r['stop_reason']}")
    print(f"Cost: ${r['cost_usd']:.6f}")
    print()

In [ ]:
QUESTION = "How long would it take to build a login page with email and password?"

ROLE = (
    "You are a senior software estimation expert. "
    "Always give an optimistic and a pessimistic estimate in hours, "
    "list the tasks, and state your assumptions."
)

# Level 1: no system prompt
level_1 = query_claude(message=QUESTION)

# Level 2: same question, with a role
level_2 = query_claude(message=QUESTION, system_prompt=ROLE)

print_result("LEVEL 1 (no role)", level_1)
print_result("LEVEL 2 (with role)", level_2)